<table>
  <tr>
    <td><div align="left"><font size="30">Cart-pole system</font></div></td>
    <td><img src="https://raw.githubusercontent.com/petercorke/bdsim/main/docs/figs/bdsim_logo.png" width="300" height="120" style="height:auto"></td>
  </tr>
</table>

**Modeling the classic cart-pole or inverted-pendulum system.**

(c) Peter Corke 2026

In [ ]:
# BDSIM_BOOTSTRAP_CELL: installs the package (Colab, JupyterLite or local), generated by sync_bootstrap.py from _bdsim_nb_bootstrap.py, do not edit

import subprocess
import sys
PACKAGE = 'bdsim'
IMPORT_NAME = 'bdsim'
GIT_URL: str | None = 'https://github.com/petercorke/bdsim'
JL_PACKAGES = []
JL_NO_DEPS: list[str] = []
JL_EXTRAS: dict[str, list[str]] = {'control': ['control']}
COLAB_PACKAGES = []

def _version_of(package: str, module) -> str:
    import importlib.metadata
    try:
        return importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        return str(getattr(module, '__version__', 'unknown'))

def _older(version: str, required: str) -> bool:
    try:
        from packaging.version import InvalidVersion, Version
        return Version(version) < Version(required)
    except (ImportError, InvalidVersion):
        return False

async def ensure_installed(extras: tuple[str, ...] | list[str]=(), packages: tuple[str, ...] | list[str]=(), min_version: str | None=None) -> bool:
    if sys.platform == 'emscripten':
        try:
            import piplite
            install = piplite.install
        except ImportError:
            import micropip
            install = micropip.install
        for name in JL_NO_DEPS:
            await install(name, deps=False)
        for name in JL_PACKAGES:
            await install(name)
        for extra in extras:
            for name in JL_EXTRAS.get(extra, []):
                await install(name)
        for name in packages:
            await install(name)
        await install(PACKAGE)
        where, colab = ('in browser', False)
    else:
        try:
            import google.colab
        except ImportError:
            where, colab = ('locally', False)
        else:
            print(f'Installing {PACKAGE}...')
            requirement = PACKAGE
            if extras:
                requirement += '[' + ','.join(extras) + ']'
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', requirement] + list(packages) + COLAB_PACKAGES, check=True)
            try:
                from matplotlib.backends import backend_registry
            except ImportError:
                pass
            else:
                clear = getattr(backend_registry, '_clear', None)
                if clear is not None:
                    clear()
            from google.colab import output
            output.enable_custom_widget_manager()
            where, colab = ('on Colab', True)
    import importlib
    module = importlib.import_module(IMPORT_NAME)
    version = _version_of(PACKAGE, module)
    if colab and min_version is not None and _older(version, min_version) and GIT_URL:
        print(f'{PACKAGE} {version} on PyPI is older than {min_version}, installing from GitHub...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', f'{PACKAGE} @ git+{GIT_URL}@main'], check=True)
        importlib.invalidate_caches()
        version = _version_of(PACKAGE, module)
        where += ', development version from GitHub'
    if not colab and where == 'locally':
        import importlib.metadata
        for name in packages:
            try:
                importlib.metadata.version(name)
            except importlib.metadata.PackageNotFoundError:
                print(f'This notebook needs {name}: pip install {name}')
    if min_version is not None and where in ('locally', 'on Colab') and _older(version, min_version):
        print(f'This notebook needs {PACKAGE} {min_version} or later: pip install -U {PACKAGE}')
    print(f'Running {where} using {PACKAGE} v{version}')
    return colab

COLAB = await ensure_installed(extras=['control'], min_version='1.5.0')


We start with the standard imports, construct a simulator object, and use that to construct
a block diagram with all the `bdsim` blocks included as methods.

In [ ]:
import math
import numpy as np
import bdsim

sim = bdsim.BDSim(animation=True, animation_rate=0.5)  # create simulator

bd = sim.blockdiagram()  # create an empty block diagram

The dynamics of the cart and pole system are taken from [Underactuated Robotics, Chapter 3, Russ Tedrake, MIT](https://underactuated.csail.mit.edu/acrobot.html#cart_pole).

A cart of mass $M$ rolls on a horizontal surface with viscous
friction $b$ and with position $x$. A horizontal force $F$ is applied to the cart in its direction of positive increase of $x$.

The pole is assumed to be a lumped mass $m$, at a distance $l$ from the pivot.  The angle of the pole is $\theta$ and increases clockwise from vertical ($\theta=0$). 

The equations of motion can be derived using any number of techniques such Lagrangian
or Newton-Euler dynamics.  For execution in a block diagram model, the key step is
isolate the second derivatives of the state variables which we feed into a chain of
integrators

$$
\begin{aligned}
\ddot{x} &= \frac{f + m  \sin \theta (l  \dot{\theta}^2 + g \cos \theta )}{D} \\
\ddot{\theta} &= \frac{-f \cos \theta - m  l  \dot{\theta}^2 \cos \theta \sin \theta - (M + m) g \sin \theta}{l D} 
\end{aligned}
$$

where $f$ is the nett force on the cart (applied minus friction) and $D=M + m  \sin^2 \theta$ is the determinant of the mass matrix.

As a sanity check we will compute the total energy of the system. The kinetic energy is

$$
T = \frac{M + m}{2}  \dot{x}^2 + \frac{m  l^2  \dot{\theta}^2}{2} + m l \dot{x}  \dot{\theta} \cos \theta
$$

and the potential energy (zero with pole vertically down) is

$$
U = -m  g  l (1+\cos \theta)
$$
and the negative sign accounts for the way gravity is defined.

The total energy of the system is $E = T + U$.

Next up, we define the parameters of the system

In [ ]:
M = 1.0  # mass of cart
m = 0.1  # mass of pendulum
l = 0.5  # length of pendulum
g = -9.81  # gravity
I = m * l**2 / 3  # inertia of pendulum about CoG  ????
b = 10  # cart friction


The system can be modeled as two chains of integrators.  The first integrates cart acceleration $\ddot{x}$ to velocity $\dot{x}$ to position $x$.  The initial position and velocity is zero.

In [ ]:

x_dot = bd.INTEGRATOR(name="x_dot")
x = x_dot >> bd.INTEGRATOR(name="x")


Note that we are using the right-shift operator to implicitly **connect** the output of the `x_dot` block to the input of the next integrator, and the result of that expression is assigned to
`x`.

The other chain integrates the pole's angular acceleration $\ddot{\theta}$ to angular
velocity $\dot{\theta}$ to angle $\theta$.  The initial angle and velocity is zero, the pole
pointing straight up

In [ ]:

theta_dot = bd.INTEGRATOR(name="theta_dot")
theta = theta_dot >> bd.INTEGRATOR(name="theta")


We use a pair of function blocks to compute the sine and cosine of the pole angle

In [ ]:
cos_theta = theta >> bd.FUNCTION(math.cos, name="cos(theta)")
sin_theta = theta >> bd.FUNCTION(math.sin, name="sin(theta)")

The system is initially in an unstable equilbrium with all states zero, the cart and pole not moving, and the pole pointing upwards.  We apply a force pulse ($t \in [1, 1.2]$) to the cart using the difference of two step functions

In [ ]:
K = np.zeros((1,4))
F = bd.PIECEWISE((0,0), (1,1), (1.2,0), name="disturbance") -  K * bd.STATE()

Next we compute the common term $D$ and the nett force on the cart

In [ ]:
D = M + m * sin_theta**2  # determinant of the mass matrix
f = F - b * x_dot  # nett force on the cart (input minus friction)

In [ ]:
x_dot[0] = (
    f + m * sin_theta * (l * theta_dot**2 + g * cos_theta)
) / D  # cart acceleration
theta_dot[0] = (
    -f * cos_theta
    - m * l * theta_dot**2 * cos_theta * sin_theta
    - (M + m) * g * sin_theta
) / (l * D)  # pendulum angular acceleration

As a cross check we compute the total energy of the system `E` and connect that to a scope.  After the force pulse (which introduces energy) we should see the total energy of the system decrease monotonically.  The kinetic energy of the pole is dissipated by the wheel friction of the cart.

In [ ]:
T = (
    1 / 2 * (M + m) * x_dot**2
    + 1 / 2 * m * l**2 * theta_dot**2
    + m * l * x_dot * theta_dot * cos_theta
)  # kinetic energy (note coupling term from cart-pole interaction)
U = -m * g * l * (1+cos_theta)  # potential energy (zero at vertical down)
E = T + U

# display the system state evolution on a scope and animation
scope = bd.SCOPE(
    nin=4,
    inputs=[F, x, theta, E],
    labels=("F", "x", r"$\theta$", "E"),
    loc="lower right",
)


To include a graphical animation of our cart and pole system we will use simple Matplotlib 
patch graphics.  This kind of code is tedious to write, so I prompted Gemini:

> *I want to write a simple matplotlib
> animation of the cart pole system. A rectangle for the cart, a horizontal line for it
> to move across, and the pendulum which pivots about the middle top of the cart. Use
> MPL patches and their transformations. For the purpose of the prototype, the cart
> position and pole angle can increase linearly with time.*

I then extracted the core part of the logic to setup the the animation

In [ ]:

import matplotlib.patches as patches
import matplotlib.transforms as transforms

def cartpole_init(self, fig, ax):
    # 1. Setup the figure and axis
    ax.set_xlim(-5, 5)
    ax.set_ylim(-2, 4)
    ax.set_aspect("equal")
    ax.grid(True, linestyle="--")

    # Draw the horizontal track
    self.ax.axhline(0, color="black", lw=1.5)

    # 2. Define geometry constraints
    cart_w, cart_h = 1.0, 0.6
    pole_w, pole_h = 0.1, 2.0

    # Pivot point relative to the cart: top-middle
    pivot_y_rel = cart_h

    # 3. Create the patches at the origin (0,0)
    # Cart anchor is bottom-left, so we center it horizontally at X=0
    cart = patches.Rectangle(
        (-cart_w / 2, 0),
        cart_w,
        cart_h,
        linewidth=1.5,
        edgecolor="blue",
        facecolor="lightblue",
    )

    # Pole anchor is bottom-left. Center it horizontally at X=0, resting at Y=0
    pole = patches.Rectangle(
        (-pole_w / 2, 0),
        pole_w,
        pole_h,
        linewidth=1.5,
        edgecolor="red",
        facecolor="salmon",
    )

    ax.add_patch(cart)
    ax.add_patch(pole)

    self.cart = cart
    self.pole = pole

    self.pivot_y_rel = pivot_y_rel

and the actual update

In [ ]:

def cartpole_update(self, t, u):
    cart_x = u[0]
    pole_angle = u[1]
    ax = self.ax

    # --- Cart Transformation ---
    # 1. Shift the cart horizontally by cart_x
    # 2. Transform from model to display coordinates (+ ax.transData)
    cart_trans = transforms.Affine2D().translate(cart_x, 0) + ax.transData
    self.cart.set_transform(cart_trans)

    # --- Pole Transformation ---
    # 1. Rotate the pole around its own bottom-center (0,0 in its local space)
    # 2. Translate it to match the cart's current absolute pivot position
    # 3. Transform from model to display coordinates (+ ax.transData)
    pivot_x_abs = cart_x
    pivot_y_abs = self.pivot_y_rel

    pole_trans = (
        transforms.Affine2D().rotate(-pole_angle).translate(pivot_x_abs, pivot_y_abs)
    ) + ax.transData

    self.pole.set_transform(pole_trans)



These two functions are passed to an `ANIMATION` block driven by the outputs of the `x` and `theta` blocks.  These inputs are the elements of `u` passed to `cartpole_update`.

In [ ]:

anim = bd.ANIMATION(
    init=cartpole_init, update=cartpole_update, nin=2, inputs=[x, theta]
)


Our system is built, we will compile it, and display a summary

In [ ]:
bd.compile()  # check the diagram
print(bd)

Which has 47 blocks, although we have only explicitly created 4 integrators, 2 functions, 2 steps, an animation and a scope.  The majority of the blocks were created by the arithmetic expressions for the equations of motion and system energy.  

Here is a full list of blocks.

In [ ]:
bd.report_summary()

where the implicitly created blocks have names prefixed by `_`.

To create all these blocks and connect them would be both tedious and error prone.  The power of `bdsim` is that it lets you move effortlessly between expression in terms of blocks or using Python expressions.

In [ ]:
out = sim.run(bd, T=20, watch=[F, x, theta])  # simulate for 5s
print(out)


In [ ]:
f,g = bd.functions(inputs=["disturbance"], outputs=[scope[1], scope[2]])

In [ ]:
X = bd.linearize(t= 0, inputs=["disturbance"], outputs=[scope[1], scope[2]], full=True)
print(X)



Now we can design a linear controller using the Python Control package

```
$ pip install control
```

In [ ]:
import control as ct

# Create the state-space system (optional, but good practice)
sys = ct.ss(X.A, X.B, X.C, X.D, states=X.xindex.names, inputs=X.uindex.names, outputs=X.yindex.names)

# 2. Define the cost matrices
# Q must be positive semi-definite (penalizes state errors)
# R must be positive definite (penalizes control effort)
Q = np.diag([1, 1, 10, 1])  # weights for [x, x_dot, theta, theta_dot]
R = np.array([[100]]) # weight for control input (force F)

# 3. Compute the LQR optimal gain matrix K
# control.lqr returns:
# K: The optimal feedback gain matrix (u = -Kx)
# S: The solution to the algebraic Riccati equation
# E: The eigenvalues (poles) of the closed-loop system (A - BK)
K, S, E = ct.lqr(sys, Q, R)

# Note: You can also pass the arrays directly without creating a sys object:
# K, S, E = ct.lqr(A, B, Q, R)

print("Optimal Feedback Gain (K): \n", K)
print("Closed-loop Poles (E): \n", E)

In [ ]:
bd["_gain.0"].K = K  # set the gain block to the computed LQR gain
out = sim.run(bd, T=20, watch=[F, x, theta])  # simulate for 5s

In [ ]:
bd["_gain.0"].K=[1,2,3,4]
bd["_gain.0"].K